# Baseline: NetworkX 2.8 Betweenness Centrality

**Repo:** `networkx/networkx`  
**Baseline commit/tag:** `networkx-2.8` (SHA `3bf243a47eb6487cea30d6978d4f09d102ce97fb`)  
**Release date:** April 11 2022 , over 12 months before submission  

## What is the slow workload?

`nx.betweenness_centrality(G)` on an undirected Barabási–Albert random graph  
with n=2000 nodes and m=6 edges per new node (~12 000 edges total).

Betweenness centrality (Brandes 2001) runs BFS + back-propagation once per  
source node O(n·m) work total. The pure-Python implementation in NetworkX 2.8  
operates on Python `dict` and `deque` objects inside those loops, incurring full  
CPython interpreter overhead for every edge visit.

## Why is this workload meaningful?

Betweenness centrality is one of the most widely used graph analysis metrics , 
social network analysis, routing, biology, citation networks. NetworkX is the  
most popular Python graph library (14 k+ GitHub stars). The slowness is a  
well-documented pain point: a 90 k-node graph takes 14 hours in NetworkX vs  
10 minutes in igraph (C backend). Our 2000-node workload represents a real  
interactiveanalysis scale where a 10 s wait kills iteration speed.

## How was the slow path discovered?

1. Ran `cProfile` on `nx.betweenness_centrality` 99 % of time in  
   `_single_source_shortest_path_basic` and `_accumulate_basic`.
2. Inspected the source: both are pure-Python loops over Python dicts/deques  
   with no vectorisation or compiled path.
3. Confirmed with open issues (e.g. gboeing/osmnx#153, NVIDIA blog 2023)  
   that this is a known bottleneck, not fixed in 2.8.
4. Verified no `--fast` flag, no faster backend, no upstream patch at this commit.

In [ ]:
# Runtime info
!cat /proc/cpuinfo | grep 'model name' | head -1
!free -h
!python --version

In [ ]:
# Clean up any previous run, then clone NetworkX 2.8 and install
!rm -rf nx_baseline
!git clone --depth 1 --branch networkx-2.8 https://github.com/networkx/networkx.git nx_baseline

# Use pip install (non-editable) which works reliably on Python 3.12 Colab
!pip install nx_baseline/ --quiet 2>&1 | grep -E 'Successfully|ERROR|error' || true

# Force the installed version onto sys.path
import importlib, sys
# Remove any cached networkx modules so the freshly installed 2.8 is picked up
to_remove = [k for k in sys.modules if k.startswith('networkx')]
for k in to_remove:
    del sys.modules[k]

import networkx as nx
print('NetworkX version:', nx.__version__)  # must be 2.8
assert nx.__version__ == '2.8', f'Wrong version: {nx.__version__}'

In [ ]:
import json, statistics, time
import networkx as nx

# Reproducible graph:same seed used in all three notebooks
SEED = 42
G = nx.barabasi_albert_graph(n=2000, m=6, seed=SEED)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# Run once to generate golden reference output
golden_bc = nx.betweenness_centrality(G)

with open('golden_output.json', 'w') as f:
    json.dump({str(k): v for k, v in golden_bc.items()}, f)

print('Golden reference saved. Top 5 nodes by centrality:')
for node, val in sorted(golden_bc.items(), key=lambda x: -x[1])[:5]:
    print(f'  node {node}: {val:.6f}')

In [ ]:
# Run the existing NetworkX test suite and record which tests pass
!pip install pytest --quiet
import subprocess

result = subprocess.run(
    ['python', '-m', 'pytest',
     'nx_baseline/networkx/algorithms/centrality/tests/test_betweenness_centrality.py',
     '-v', '--tb=no', '--no-header', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])

passing = [line.split(' ')[0] for line in result.stdout.split('\n') if ' PASSED' in line]
with open('baseline_passing_tests.json', 'w') as f:
    json.dump(passing, f)
print(f'Saved {len(passing)} passing tests.')

In [ ]:
# Baseline timing: 2 warmup runs + 7 measured runs
N_WARMUP  = 2
N_MEASURE = 7

print('Warming up...')
for _ in range(N_WARMUP):
    nx.betweenness_centrality(G)

print('Measuring...')
times = []
for i in range(N_MEASURE):
    t0 = time.perf_counter()
    nx.betweenness_centrality(G)
    elapsed = time.perf_counter() - t0
    times.append(elapsed)
    print(f'  run {i+1}: {elapsed:.3f}s')

times_sorted = sorted(times)
med = statistics.median(times)
iqr = times_sorted[5] - times_sorted[1]

print(f'\nBaseline median: {med:.3f}s')
print(f'Baseline IQR:    {iqr:.3f}s')

with open('baseline_timing.json', 'w') as f:
    json.dump({'median': med, 'iqr': iqr, 'all_times': times,
               'n_warmup': N_WARMUP, 'n_measured': N_MEASURE}, f)
print('Timing saved to baseline_timing.json')